<a href="https://colab.research.google.com/github/jhonatanyara/TAM-Laura-Sof-a-Casta-o-Pineda-Jhonatan-Yara-Lopez-/blob/main/Taller_1_TAM_2026_2_Solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller 1: Teoría de Aprendizaje de Máquina (2026-2)
**Profesor:** Diego Armando Pérez Rosero  
**Tema:** Clasificación supervisada, vecinos cercanos y evaluación de modelos  

Este cuaderno contiene la implementación reproducible en Python de los puntos de programación y la verificación numérica y gráfica de los cálculos manuales del taller, utilizando una semilla fija (`random_state=42`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

# Semilla fija para reproducibilidad y configuración de gráficas
SEED = 42
np.random.seed(SEED)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
print("Entorno configurado correctamente. Semilla fija =", SEED)

---
## Punto 1: Clasificación lineal y algoritmo del perceptrón

### 1.1 Frontera de decisión y clasificación de observaciones
Con $w = [1, 2]^\top$ y $b = -3$, la frontera de decisión es $w^\top x + b = 0$, es decir:
$$x_{(1)} + 2x_{(2)} - 3 = 0 \implies x_{(2)} = -0.5x_{(1)} + 1.5$$

Evaluamos los puntos $x_1=(0,0)$, $x_2=(1,0)$, $x_3=(2,1)$ y $x_4=(3,1)$ usando la regla de predicción:
$$\hat{y} = \begin{cases} +1, & w^\top x + b > 0 \\ -1, & w^\top x + b \le 0 \end{cases}$$

In [ ]:
# Parámetros del clasificador lineal
w = np.array([1.0, 2.0])
b = -3.0

puntos = {
    'x1': np.array([0.0, 0.0]),
    'x2': np.array([1.0, 0.0]),
    'x3': np.array([2.0, 1.0]),
    'x4': np.array([3.0, 1.0])
}

def predecir_perceptron(x, w, b):
    activacion = float(np.dot(w, x) + b)
    y_hat = 1 if activacion > 0 else -1
    return activacion, y_hat

print("Resultados Punto 1.1:")
for nombre, x_val in puntos.items():
    act, pred = predecir_perceptron(x_val, w, b)
    print(f"  {nombre} = {tuple(x_val.astype(int))} | w^T x + b = {act:5.1f} -> Clase predicha: {pred:+d}")

# Gráfica de la frontera de decisión y las observaciones
fig, ax = plt.subplots(figsize=(7.5, 5))
x1_vals = np.linspace(-0.8, 4.0, 200)
x2_vals = (-w[0] * x1_vals - b) / w[1]

ax.plot(x1_vals, x2_vals, 'k--', linewidth=2, label=r'Frontera: $x_{(1)} + 2x_{(2)} - 3 = 0$')
ax.fill_between(x1_vals, x2_vals, 3.2, color='royalblue', alpha=0.14, label=r'Región $+1$ ($w^\top x + b > 0$)')
ax.fill_between(x1_vals, -1.0, x2_vals, color='crimson', alpha=0.14, label=r'Región $-1$ ($w^\top x + b \leq 0$)')

for nombre, x_val in puntos.items():
    _, pred = predecir_perceptron(x_val, w, b)
    color = 'royalblue' if pred == 1 else 'crimson'
    marker = 'o' if pred == 1 else 's'
    ax.scatter(x_val[0], x_val[1], color=color, s=100, marker=marker, edgecolors='black', zorder=5)
    ax.annotate(
        f"{nombre} {tuple(x_val.astype(int))}\n($\\hat{{y}}={pred:+d}$)",
        xy=(x_val[0], x_val[1]),
        xytext=(x_val[0] + 0.12, x_val[1] + 0.12),
        fontsize=9,
        fontweight='bold'
    )

# Vector normal w desde el punto (1, 1) sobre la recta
ax.quiver(1, 1, w[0], w[1], angles='xy', scale_units='xy', scale=2.5,
          color='darkgreen', width=0.008, label=r'Vector normal $w = [1, 2]^\top$')

ax.set_xlim(-0.6, 4.0)
ax.set_ylim(-0.8, 2.8)
ax.set_xlabel('$x_{(1)}$')
ax.set_ylabel('$x_{(2)}$')
ax.set_title('Punto 1.1: Frontera de decisión del clasificador lineal')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 1.3 Verificación numérica de la actualización del perceptrón
Con $w = [0, 0]^\top$, $b = 0$, $\eta = 0.5$, $x = [2, 1]^\top$ y $y = +1$, verificamos si corresponde actualizar y comparamos el nuevo margen calculado directamente contra la expresión demostrada $m_{\text{nuevo}} = m + \eta(\|x\|_2^2 + 1)$.

In [ ]:
w_init = np.array([0.0, 0.0])
b_init = 0.0
eta = 0.5
x_obs = np.array([2.0, 1.0])
y_obs = 1

act_init, y_hat_init = predecir_perceptron(x_obs, w_init, b_init)
m_init = y_obs * act_init

print(f"Activación inicial: w^T x + b = {act_init:.2f}")
print(f"Predicción inicial: y_hat = {y_hat_init:+d} | Etiqueta real: y = {y_obs:+d}")
print(f"Margen inicial:     m = {m_init:.2f}")

if y_hat_init != y_obs:
    print("\n-> Error de clasificación detectado: SÍ corresponde actualizar.")
    w_nuevo = w_init + eta * y_obs * x_obs
    b_nuevo = b_init + eta * y_obs
    m_nuevo_directo = y_obs * (np.dot(w_nuevo, x_obs) + b_nuevo)
    m_nuevo_formula = m_init + eta * (np.linalg.norm(x_obs)**2 + 1)

    print(f"w_nuevo = {w_nuevo}")
    print(f"b_nuevo = {b_nuevo:.2f}")
    print(f"Margen nuevo (cálculo directo):   {m_nuevo_directo:.2f}")
    print(f"Margen nuevo (fórmula teórica):   {m_nuevo_formula:.2f}")
else:
    print("\n-> Clasificación correcta: NO corresponde actualizar.")

---
## Punto 2: Clasificación por los $k$ vecinos más cercanos ($k$-NN)

### 2.1 Verificación del cálculo manual de distancias
Calculamos la distancia euclidiana desde $q = (1, 1)$ hacia las cinco observaciones de entrenamiento y evaluamos la votación mayoritaria para $k \in \{1, 3, 5\}$.

In [ ]:
q = np.array([1.0, 1.0])
X_manual = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [3.0, 1.0],
    [0.0, 3.0],
    [1.1, 1.1]
])
clases_manual = ['A', 'A', 'B', 'B', 'B']
nombres_obs = ['x1', 'x2', 'x3', 'x4', 'x5']

distancias = np.linalg.norm(X_manual - q, axis=1)
orden = np.argsort(distancias)

print("Observaciones ordenadas de la más cercana a la más lejana respecto a q=(1,1):")
for pos, idx in enumerate(orden, start=1):
    print(f"  {pos}. {nombres_obs[idx]} = {tuple(X_manual[idx])} | d = {distancias[idx]:.4f} | Clase = {clases_manual[idx]}")

print("\nClasificación por votación mayoritaria:")
for k in [1, 3, 5]:
    vecinos = [clases_manual[idx] for idx in orden[:k]]
    votos_A = vecinos.count('A')
    votos_B = vecinos.count('B')
    pred = 'A' if votos_A > votos_B else 'B'
    print(f"  k = {k}: vecinos = {vecinos} (A={votos_A}, B={votos_B}) -> Predicción: Clase {pred}")

### 2.2 Implementación de $k$-NN sobre `make_moons`
Generamos $N = 300$ observaciones bidimensionales con `make_moons(noise=0.20, random_state=42)`, separamos 70% entrenamiento y 30% prueba con estratificación, entrenamos `KNeighborsClassifier` para $k \in \{1, 5, 15\}$, reportamos la exactitud en ambos conjuntos y graficamos los datos junto con las regiones de decisión.

In [ ]:
# 1. Generación de datos y partición estratificada 70% / 30%
X, y = make_moons(n_samples=300, noise=0.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)

print(f"Tamaño total: {len(X)} | Entrenamiento: {len(X_train)} | Prueba: {len(X_test)}")

# 2. Entrenamiento y evaluación para k in {1, 5, 15}
k_values = [1, 5, 15]
modelos_knn = {}

print("\nExactitud (Accuracy) por valor de k:")
print("-" * 55)
print(f"{'k':>4} | {'Acc. Entrenamiento':>20} | {'Acc. Prueba':>20}")
print("-" * 55)
for k in k_values:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, clf.predict(X_train))
    acc_test = accuracy_score(y_test, clf.predict(X_test))
    modelos_knn[k] = (clf, acc_train, acc_test)
    print(f"{k:>4d} | {acc_train:>12.4f} ({acc_train*100:5.2f}%) | {acc_test:>12.4f} ({acc_test*100:5.2f}%)")
print("-" * 55)

# 3. Gráfica de los datos originales (entrenamiento y prueba)
fig, ax = plt.subplots(figsize=(6.8, 4.6))
cmap_bold = ['#D9534F', '#0275D8']
for clase, color in zip([0, 1], cmap_bold):
    ax.scatter(
        X_train[y_train == clase, 0], X_train[y_train == clase, 1],
        c=color, s=30, alpha=0.8, edgecolors='k', linewidth=0.5, label=f'Train Clase {clase}'
    )
    ax.scatter(
        X_test[y_test == clase, 0], X_test[y_test == clase, 1],
        c=color, marker='^', s=45, edgecolors='k', linewidth=0.8, label=f'Test Clase {clase}'
    )
ax.set_title('Conjunto de datos make_moons (N=300, noise=0.20)')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend(loc='upper right', fontsize=8.5)
plt.tight_layout()
plt.show()

# 4. Gráfica de regiones de decisión para k = 1, 5 y 15
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
cmap_light = ListedColormap(['#FFBBBB', '#BBBBFF'])

x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

for ax, k in zip(axes, k_values):
    clf, acc_tr, acc_te = modelos_knn[k]
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.65)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8, alpha=0.55)

    for clase, color in zip([0, 1], cmap_bold):
        ax.scatter(
            X_train[y_train == clase, 0], X_train[y_train == clase, 1],
            c=color, s=24, alpha=0.85, edgecolors='k', linewidth=0.5, label=f'Train {clase}'
        )
        ax.scatter(
            X_test[y_test == clase, 0], X_test[y_test == clase, 1],
            c=color, marker='^', s=42, edgecolors='k', linewidth=0.8, label=f'Test {clase}'
        )

    ax.set_title(f'k-NN (k = {k})\nTrain: {acc_tr:.4f} | Test: {acc_te:.4f}')
    ax.set_xlabel('$x_1$')
    if k == 1:
        ax.set_ylabel('$x_2$')
        ax.legend(loc='upper right', fontsize=8)

plt.suptitle('Punto 2.2: Regiones de decisión en make_moons para k = 1, 5 y 15', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2.3 Interpretación de resultados ($k$-NN)
- **$k$ muy pequeño ($k = 1$):** El modelo memoriza exactamente los datos de entrenamiento (`1.0000` en entrenamiento), produciendo fronteras muy irregulares con "islas" alrededor de puntos ruidosos (alta varianza / sobreajuste), lo que reduce la exactitud en prueba a `0.9333`.
- **$k$ muy grande ($k = 15$):** La frontera se suaviza en exceso, aumentando el sesgo y perdiendo parte de la curvatura local en los extremos de las dos lunas, lo que también reduce la exactitud en prueba a `0.9333`.
- **Mejor generalización:** En este experimento, **$k = 5$** generalizó mejor, obteniendo la mayor exactitud en prueba (`0.9556`) y la menor brecha entre entrenamiento (`0.9571`) y prueba (`0.9556`), logrando un buen equilibrio entre sesgo y varianza.
- **Selección con validación, no con prueba:** Escoger $k$ usando el conjunto de prueba introduce fuga de información (*data leakage*) y sesga de forma optimista la estimación del desempeño real. La selección formal de hiperparámetros debe hacerse mediante un conjunto de validación o validación cruzada (*k-fold*) sobre el entrenamiento, reservando el conjunto de prueba únicamente para la evaluación final.

---
## Punto 3: Evaluación de un clasificador binario

### 3.3 Verificación en Python de la matriz de confusión y las métricas
Verificamos los cálculos manuales con `confusion_matrix`, `accuracy_score`, `precision_score`, `recall_score` y `f1_score` de `scikit-learn`.

In [ ]:
y_true = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
y_pred = np.array([0, 1, 0, 0, 1, 0, 1, 1, 0, 1])

# Matriz de confusión: filas = clases reales, columnas = clases predichas
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Matriz de confusión (Filas = Real [0, 1], Columnas = Predicho [0, 1]):")
print(cm)
print(f"\nConteos -> TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")
print(f"Accuracy  = {acc:.4f}  (7/10)")
print(f"Precision = {prec:.4f}  (4/5)")
print(f"Recall    = {rec:.4f}  (4/6)")
print(f"F1-Score  = {f1:.4f}  (8/11)")

# Visualización de la matriz de confusión
fig, ax = plt.subplots(figsize=(5.2, 4.2))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Clase 0 (Neg)', 'Clase 1 (Pos)'])
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Punto 3: Matriz de Confusión')
ax.grid(False)
plt.tight_layout()
plt.show()

### Discusión: Dos situaciones en las que la exactitud (*Accuracy*) por sí sola resulta engañosa
1. **Clases fuertemente desbalanceadas:** En problemas como detección de fraude bancario (por ejemplo, 99.5% transacciones legítimas y 0.5% fraudulentas), un modelo trivial que predice siempre la clase mayoritaria obtiene un 99.5% de exactitud, pero su *Recall* sobre la clase positiva es 0%, por lo que no detecta ningún fraude.
2. **Costos asimétricos de los errores (FP vs. FN):** En diagnóstico clínico de enfermedades graves o en sistemas de seguridad crítica, omitir un caso positivo real (Falso Negativo) tiene un costo mucho mayor que generar una falsa alarma (Falso Positivo). Dos clasificadores pueden compartir exactamente el mismo *Accuracy*, pero comportarse de forma completamente distinta en *Recall* y *Precision*, haciendo indispensable evaluar estas métricas conjuntamente.